In [0]:
import os
import json
import gradio as gr
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# ============= CONFIGURATION =============
LLM_ENDPOINT = "databricks-llama-4-maverick"  # Working endpoint!
SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")

# Load pre-aggregated mandi data (NO SQL CONNECTION NEEDED!)
MANDI_DATA = []
try:
    data_path = os.path.join(os.path.dirname(__file__), "mandi_data.json")
    with open(data_path, 'r') as f:
        MANDI_DATA = json.load(f)
    print(f"✅ Loaded {len(MANDI_DATA)} mandi records from cache")
except Exception as e:
    print(f"⚠️ Could not load mandi data: {e}")

# Initialize Databricks client
try:
    w = WorkspaceClient()
    print("✅ Databricks client initialized")
except Exception as e:
    print(f"⚠️ Databricks client initialization failed: {e}")
    w = None

LANG_REPLY_INSTRUCTION = {
    "English": "Reply in English.",
    "Hinglish": "Reply in Romanized Hindi (Latin script).",
    "हिन्दी": "Reply in Hindi (Devanagari script).",
    "தமிழ்": "Reply in Tamil.",
    "తెలుగు": "Reply in Telugu.",
    "मराठी": "Reply in Marathi.",
    "বাংলা": "Reply in Bengali.",
    "ਪੰਜਾਬੀ": "Reply in Punjabi (Gurmukhi script).",
}

# ============= UI TRANSLATIONS =============
UI_TRANSLATIONS = {
    "English": {
        "title": "🌾 Mandi Mitra — AI Agricultural Advisor",
        "subtitle": "*Ask about crop prices and best mandis in any Indian language!*",
        "powered_by": "<small>Powered by: Llama 4 Maverick + Pre-computed Mandi Data</small>",
        "lang_label": "Language",
        "district_label": "District",
        "quintals_label": "Quintals",
        "question_label": "⌨️ Type your question",
        "question_placeholder": "Where should I sell onions?",
        "submit_button": "Ask Mandi Mitra ➤",
        "response_label": "🤖 Mandi Mitra says",
        "examples_header": "### Example Questions:",
        "example_1": "- Where should I sell onions in my district?",
        "example_2": "- What is the current tomato price in Nashik?",
        "example_3": "- Best market for potatoes near me?",
    },
    "Hinglish": {
        "title": "🌾 Mandi Mitra — AI Kheti Salahkar",
        "subtitle": "*Kisi bhi Indian bhasha mein fasal ki keemat aur best mandi ke baare mein poochiye!*",
        "powered_by": "<small>Powered by: Llama 4 Maverick + Pre-computed Mandi Data</small>",
        "lang_label": "Bhasha / Language",
        "district_label": "Zila / District",
        "quintals_label": "Quintal",
        "question_label": "⌨️ Apna sawal likhiye",
        "question_placeholder": "Pyaaz kahan bechun?",
        "submit_button": "Mandi Mitra se Poochiye ➤",
        "response_label": "🤖 Mandi Mitra ka Jawab",
        "examples_header": "### Example Sawal:",
        "example_1": "- Mujhe pyaaz kahan bechna chahiye?",
        "example_2": "- Nashik mein tamatar ka price kya hai?",
        "example_3": "- Aaloo ke liye best mandi konsi hai?",
    },
    "हिन्दी": {
        "title": "🌾 मंडी मित्र — एआई कृषि सलाहकार",
        "subtitle": "*किसी भी भारतीय भाषा में फसल की कीमत और सर्वोत्तम मंडी के बारे में पूछें!*",
        "powered_by": "<small>संचालित: Llama 4 Maverick + पूर्व-संगणित मंडी डेटा</small>",
        "lang_label": "भाषा",
        "district_label": "जिला",
        "quintals_label": "क्विंटल",
        "question_label": "⌨️ अपना प्रश्न लिखें",
        "question_placeholder": "प्याज कहाँ बेचूं?",
        "submit_button": "मंडी मित्र से पूछें ➤",
        "response_label": "🤖 मंडी मित्र का उत्तर",
        "examples_header": "### उदाहरण प्रश्न:",
        "example_1": "- मुझे प्याज कहाँ बेचना चाहिए?",
        "example_2": "- नासिक में टमाटर की कीमत क्या है?",
        "example_3": "- आलू के लिए सबसे अच्छी मंडी कौन सी है?",
    },
    "தமிழ்": {
        "title": "🌾 மண்டி மித்ரா — AI விவசாய ஆலோசகர்",
        "subtitle": "*எந்த இந்திய மொழியிலும் பயிர் விலைகள் மற்றும் சிறந்த மண்டிகள் பற்றி கேளுங்கள்!*",
        "powered_by": "<small>இயக்குபவர்: Llama 4 Maverick + முன்கணிக்கப்பட்ட மண்டி தரவு</small>",
        "lang_label": "மொழி",
        "district_label": "மாவட்டம்",
        "quintals_label": "குவிண்டால்கள்",
        "question_label": "⌨️ உங்கள் கேள்வியை தட்டச்சு செய்யவும்",
        "question_placeholder": "வெங்காயத்தை எங்கே விற்க வேண்டும்?",
        "submit_button": "மண்டி மித்ராவிடம் கேளுங்கள் ➤",
        "response_label": "🤖 மண்டி மித்ரா கூறுகிறார்",
        "examples_header": "### உதாரண கேள்விகள்:",
        "example_1": "- நான் வெங்காயத்தை எங்கே விற்க வேண்டும்?",
        "example_2": "- நாசிக்கில் தக்காளி விலை என்ன?",
        "example_3": "- உருளைக்கிழங்குக்கு சிறந்த சந்தை எது?",
    },
    "తెలుగు": {
        "title": "🌾 మండి మిత్ర — AI వ్యవసాయ సలహాదారు",
        "subtitle": "*ఏ భారతీయ భాషలోనైనా పంట ధరలు మరియు ఉత్తమ మండీల గురించి అడగండి!*",
        "powered_by": "<small>శక్తిచేత: Llama 4 Maverick + ముందస్తు మండి డేటా</small>",
        "lang_label": "భాష",
        "district_label": "జిల్లా",
        "quintals_label": "క్వింటాల్స్",
        "question_label": "⌨️ మీ ప్రశ్నను టైప్ చేయండి",
        "question_placeholder": "ఉల్లిపాయలను ఎక్కడ అమ్మాలి?",
        "submit_button": "మండి మిత్రను అడగండి ➤",
        "response_label": "🤖 మండి మిత్ర చెప్పేది",
        "examples_header": "### ఉదాహరణ ప్రశ్నలు:",
        "example_1": "- నేను ఉల్లిపాయలను ఎక్కడ అమ్మాలి?",
        "example_2": "- నాసిక్‌లో టమోటా ధర ఎంత?",
        "example_3": "- బంగాళదుంపల కోసం ఉత్తమ మార్కెట్ ఏది?",
    },
    "मराठी": {
        "title": "🌾 मंडी मित्र — AI कृषी सल्लागार",
        "subtitle": "*कोणत्याही भारतीय भाषेत पिकांच्या किमती आणि सर्वोत्तम मंडी बद्दल विचारा!*",
        "powered_by": "<small>चालविले: Llama 4 Maverick + पूर्व-गणना केलेला मंडी डेटा</small>",
        "lang_label": "भाषा",
        "district_label": "जिल्हा",
        "quintals_label": "क्विंटल",
        "question_label": "⌨️ तुमचा प्रश्न टाइप करा",
        "question_placeholder": "कांदे कुठे विकावे?",
        "submit_button": "मंडी मित्राला विचारा ➤",
        "response_label": "🤖 मंडी मित्र सांगतो",
        "examples_header": "### उदाहरण प्रश्न:",
        "example_1": "- मला कांदे कुठे विकावे?",
        "example_2": "- नाशिकमध्ये टोमॅटोची किंमत किती आहे?",
        "example_3": "- बटाट्यांसाठी सर्वोत्तम मार्केट कोणती?",
    },
    "বাংলা": {
        "title": "🌾 মান্ডি মিত্র — AI কৃষি পরামর্শদাতা",
        "subtitle": "*যেকোনো ভারতীয় ভাষায় ফসলের দাম এবং সেরা মান্ডি সম্পর্কে জিজ্ঞাসা করুন!*",
        "powered_by": "<small>চালিত: Llama 4 Maverick + পূর্ব-গণনা করা মান্ডি ডেটা</small>",
        "lang_label": "ভাষা",
        "district_label": "জেলা",
        "quintals_label": "কুইন্টাল",
        "question_label": "⌨️ আপনার প্রশ্ন টাইপ করুন",
        "question_placeholder": "পেঁয়াজ কোথায় বিক্রি করব?",
        "submit_button": "মান্ডি মিত্রকে জিজ্ঞাসা করুন ➤",
        "response_label": "🤖 মান্ডি মিত্র বলছেন",
        "examples_header": "### উদাহরণ প্রশ্ন:",
        "example_1": "- আমি পেঁয়াজ কোথায় বিক্রি করব?",
        "example_2": "- নাসিকে টমেটোর দাম কত?",
        "example_3": "- আলুর জন্য সেরা বাজার কোনটি?",
    },
    "ਪੰਜਾਬੀ": {
        "title": "🌾 ਮੰਡੀ ਮਿੱਤਰ — AI ਖੇਤੀ ਸਲਾਹਕਾਰ",
        "subtitle": "*ਕਿਸੇ ਵੀ ਭਾਰਤੀ ਭਾਸ਼ਾ ਵਿੱਚ ਫਸਲਾਂ ਦੀਆਂ ਕੀਮਤਾਂ ਅਤੇ ਸਭ ਤੋਂ ਵਧੀਆ ਮੰਡੀਆਂ ਬਾਰੇ ਪੁੱਛੋ!*",
        "powered_by": "<small>ਸੰਚਾਲਿਤ: Llama 4 Maverick + ਪਹਿਲਾਂ ਤੋਂ ਗਣਨਾ ਕੀਤਾ ਮੰਡੀ ਡੇਟਾ</small>",
        "lang_label": "ਭਾਸ਼ਾ",
        "district_label": "ਜ਼ਿਲ੍ਹਾ",
        "quintals_label": "ਕੁਇੰਟਲ",
        "question_label": "⌨️ ਆਪਣਾ ਸਵਾਲ ਟਾਈਪ ਕਰੋ",
        "question_placeholder": "ਪਿਆਜ਼ ਕਿੱਥੇ ਵੇਚਣੇ ਚਾਹੀਦੇ ਹਨ?",
        "submit_button": "ਮੰਡੀ ਮਿੱਤਰ ਨੂੰ ਪੁੱਛੋ ➤",
        "response_label": "🤖 ਮੰਡੀ ਮਿੱਤਰ ਕਹਿੰਦਾ ਹੈ",
        "examples_header": "### ਉਦਾਹਰਨ ਸਵਾਲ:",
        "example_1": "- ਮੈਂ ਪਿਆਜ਼ ਕਿੱਥੇ ਵੇਚਣੇ ਚਾਹੀਦੇ ਹਨ?",
        "example_2": "- ਨਾਸਿਕ ਵਿੱਚ ਟਮਾਟਰ ਦੀ ਕੀਮਤ ਕੀ ਹੈ?",
        "example_3": "- ਆਲੂਆਂ ਲਈ ਸਭ ਤੋਂ ਵਧੀਆ ਮੰਡੀ ਕਿਹੜੀ ਹੈ?",
    },
}

# ============= HELPER FUNCTIONS =============

def llm_query(messages, timeout=30):
    """Query LLM with timeout and error handling"""
    if not w:
        return "LLM service unavailable"
    
    try:
        # Convert dict messages to ChatMessage objects
        chat_messages = []
        for msg in messages:
            role = ChatMessageRole.SYSTEM if msg["role"] == "system" else ChatMessageRole.USER
            chat_messages.append(ChatMessage(role=role, content=msg["content"]))
        
        response = w.serving_endpoints.query(
            name=LLM_ENDPOINT,
            messages=chat_messages,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"LLM query error: {e}")
        return f"LLM Error: {str(e)}"

def query_mandi_data(crop, district, limit=3):
    """Query pre-loaded mandi data (NO DATABASE CONNECTION!)"""
    crop = crop.lower().strip()
    district = district.lower().strip()
    
    # Filter data
    results = [
        r for r in MANDI_DATA 
        if r.get("commodity", "").lower() == crop 
        and r.get("district_name", "").lower() == district
    ]
    
    # Sort by avg_price descending and take top N
    results = sorted(results, key=lambda x: x.get("avg_price", 0), reverse=True)[:limit]
    
    return results

def extract_intent(user_query):
    """Extract crop and district from user query using LLM"""
    sys_prompt = """You are a JSON extractor for an Indian agriculture app.
From the user's message (in ANY language):
1. Extract the crop name. Translate to English lowercase. Valid: tomato, onion, potato, wheat. Else null.
2. Extract the district in English lowercase. If not mentioned, null.
Return ONLY valid JSON: {"crop":"...","district":"..."}"""
    
    try:
        raw = llm_query([
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_query}
        ], timeout=15)
        
        # Clean up response
        raw = raw.strip().replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Intent extraction error: {e}")
        return {"crop": None, "district": None}

def normalize_district(raw_district, lang):
    """Normalize district name"""
    if lang in ("English", "Hinglish"):
        return raw_district.strip().lower()
    
    try:
        result = llm_query([
            {"role": "system", "content": "Translate this Indian district name to English. Return ONLY the English district name in lowercase, nothing else."},
            {"role": "user", "content": raw_district}
        ], timeout=10)
        return result.strip().lower()
    except:
        return raw_district.strip().lower()

def generate_reply(user_query, data_context, lang):
    """Generate natural language response"""
    lang_instruction = LANG_REPLY_INSTRUCTION.get(lang, "Reply in the same language as the user.")
    sys_prompt = f"""You are Mandi Mitra, a friendly Indian agricultural advisor.
Data context:
{data_context}

Rules:
- {lang_instruction}
- Be concise (2-4 sentences). Use ₹ for prices.
- If data is missing, say which crops are supported: tomato, onion, potato, wheat.
- End with "Jai Kisan!" or its equivalent in the reply language."""
    
    try:
        return llm_query([
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_query}
        ])
    except Exception as e:
        return f"⚠️ Response generation error: {e}"

def get_mandi_predictions(crop, district, quintals=10):
    """Get mandi predictions from cached data"""
    results = query_mandi_data(crop, district, limit=3)
    
    if not results:
        return None
    
    # Add net return calculation
    for r in results:
        r["net_return"] = (r["avg_price"] * quintals) - 250
    
    return results

# ============= MAIN CHATBOT LOGIC =============

def chatbot_logic(user_query, raw_district, quintals, lang):
    """Main chatbot logic - NO SQL CONNECTIONS!"""
    if not user_query or not user_query.strip():
        return "Please type a question."
    
    # Step 1: Normalize district
    district_en = normalize_district(raw_district, lang)
    
    # Step 2: Extract intent (crop, district)
    intent = extract_intent(user_query)
    
    if not intent or not intent.get("crop"):
        ctx = "No specific crop detected. Supported crops: tomato, onion, potato, wheat."
        return generate_reply(user_query, ctx, lang)
    
    crop = intent["crop"]
    district = intent.get("district") or district_en
    
    # Step 3: Query cached data
    results = get_mandi_predictions(crop, district, quintals)
    
    if not results:
        ctx = f"Crop: {crop}, District: {district}. No data found in our records."
    else:
        top = results[0]
        names = ", ".join([r["market_center"].title() for r in results])
        ctx = (f"Crop: {crop}, District: {district}\n"
               f"Top mandi: {top['market_center'].title()} at ₹{round(top['avg_price'], 2)}/quintal\n"
               f"Estimated profit for {quintals}q: ₹{round(top['net_return'], 2)}\n"
               f"Options: {names}")
    
    # Step 4: Generate natural language reply
    return generate_reply(user_query, ctx, lang)

# ============= UI UPDATE FUNCTION =============

def update_ui_language(selected_lang):
    """Update all UI elements based on selected language"""
    translations = UI_TRANSLATIONS.get(selected_lang, UI_TRANSLATIONS["English"])
    
    return [
        gr.update(label=translations["lang_label"]),  # lang dropdown
        gr.update(label=translations["district_label"]),  # district textbox
        gr.update(label=translations["quintals_label"]),  # quintals number
        gr.update(label=translations["question_label"], placeholder=translations["question_placeholder"]),  # text_in
        gr.update(value=translations["submit_button"]),  # submit button
        gr.update(label=translations["response_label"]),  # reply_out
    ]

# ============= GRADIO UI =============

def handle_text(text, district, quintals, lang):
    """Handle text input"""
    try:
        return chatbot_logic(text, district, int(quintals), lang)
    except Exception as e:
        return f"❌ Error: {str(e)}"

with gr.Blocks(title="🌾 Mandi Mitra", theme=gr.themes.Soft()) as demo:
    # Dynamic title and subtitle (updated via JavaScript)
    title_md = gr.Markdown("## 🌾 Mandi Mitra — AI Agricultural Advisor")
    subtitle_md = gr.Markdown("*Ask about crop prices and best mandis in any Indian language!*")
    powered_md = gr.Markdown("<small>Powered by: Llama 4 Maverick + Pre-computed Mandi Data</small>")
    
    with gr.Row():
        lang = gr.Dropdown(
            choices=list(LANG_REPLY_INSTRUCTION.keys()), 
            value="हिन्दी", 
            label="भाषा / Language"
        )
        district = gr.Textbox(value="pune", label="जिला / District")
        quintals = gr.Number(value=10, label="क्विंटल / Quintals", minimum=1)
    
    text_in = gr.Textbox(
        label="⌨️ अपना प्रश्न लिखें", 
        placeholder="प्याज कहाँ बेचूं?",
        lines=2
    )
    submit_btn = gr.Button("मंडी मित्र से पूछें ➤", variant="primary")
    reply_out = gr.Textbox(label="🤖 मंडी मित्र का उत्तर", interactive=False, lines=6)
    
    examples_md = gr.Markdown("### उदाहरण प्रश्न:")
    example1_md = gr.Markdown("- मुझे प्याज कहाँ बेचना चाहिए?")
    example2_md = gr.Markdown("- नासिक में टमाटर की कीमत क्या है?")
    example3_md = gr.Markdown("- आलू के लिए सबसे अच्छी मंडी कौन सी है?")
    
    # Language change updates UI
    lang.change(
        fn=update_ui_language,
        inputs=[lang],
        outputs=[lang, district, quintals, text_in, submit_btn, reply_out]
    )
    
    # Also update title/subtitle/examples when language changes
    def update_text_elements(selected_lang):
        translations = UI_TRANSLATIONS.get(selected_lang, UI_TRANSLATIONS["English"])
        return [
            gr.update(value=f"## {translations['title']}"),
            gr.update(value=translations['subtitle']),
            gr.update(value=translations['powered_by']),
            gr.update(value=translations['examples_header']),
            gr.update(value=translations['example_1']),
            gr.update(value=translations['example_2']),
            gr.update(value=translations['example_3']),
        ]
    
    lang.change(
        fn=update_text_elements,
        inputs=[lang],
        outputs=[title_md, subtitle_md, powered_md, examples_md, example1_md, example2_md, example3_md]
    )
    
    # Submit button action
    submit_btn.click(
        fn=handle_text,
        inputs=[text_in, district, quintals, lang],
        outputs=[reply_out]
    )

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8000)